In [1]:
import pandas as pd
import geopandas as gpd
from dask.dataframe.io.tests.test_orc import columns

In [ ]:
terrain = gpd.read_parquet('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_terrain_attributes.parquet')

In [2]:
filtered = terrain[
    (terrain["atl03_cnf"] == 4) &
    (terrain["atl08_class"].isin([1]))
].copy()

In [3]:
filtered.to_parquet('/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_terrain_ground_only.parquet')

In [2]:
geomorphon_gdf = gpd.read_parquet("/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons.parquet")




In [3]:
filtered_1 = geomorphon_gdf[
    (geomorphon_gdf["atl03_cnf"] == 4) &
    (geomorphon_gdf["atl08_class"].isin([1]))
    ].copy()


In [4]:
filtered_1.to_parquet(
    '/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_geomorphons_ground_only.parquet')

In [2]:
hand_gdf = gpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND.parquet")



In [3]:
filtered_2 = hand_gdf[
    (hand_gdf["atl03_cnf"] == 4) &
    (hand_gdf["atl08_class"].isin([1]))
    ].copy()

In [4]:
filtered_2.to_parquet(
    '/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_HAND_ground_only.parquet')

In [ ]:
import dask
import dask_geopandas as dgpd
from dask.distributed import Client
client = Client()
dask.config.set(scheduler="threads")
needed_hand = [
    'time', 'hand_alos_dem', 'hand_aster_dem', 'hand_copernicus_dеm',
    'hand_fab_dem', 'hand_nasa_dem', 'hand_srtm_dem', 'hand_tan_dem'
]
needed_geomorph = [
    'time', 'alos_dem_geomorphon', 'alos_dem_landform', 'aster_dem_geomorphon', 'aster_dem_landform',
    'copernicus_dеm_geomorphon', 'copernicus_dеm_landform', 'fab_dem_geomorphon', 'fab_dem_landform',
    'nasa_dem_geomorphon', 'nasa_dem_landform', 'srtm_dem_geomorphon', 'srtm_dem_landform',
    'tan_dem_geomorphon', 'tan_dem_landform'
]

terrain = dgpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/filter_ground/icesat2_with_terrain_ground_only.parquet").set_index('time')

geomorph = dgpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/filter_ground/icesat2_with_geomorphons_ground_only.parquet", columns=needed_geomorph).set_index('time')
hand = dgpd.read_parquet(
    "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/filter_ground/icesat2_with_HAND_ground_only.parquet", columns=needed_hand).set_index('time')


# 2. Нові колонки
geom_cols = [c for c in geomorph.columns if c not in terrain.columns and c != 'geometry']
hand_cols = [c for c in hand.columns if c not in terrain.columns and c != 'geometry']
# 3. Join-им лінено
result = terrain

if geom_cols:
    result = result.join(geomorph[geom_cols], how='left')
if hand_cols:
    result = result.join(hand[hand_cols], how='left')



from dask.diagnostics import ProgressBar

with ProgressBar():
    result.to_parquet(
        "/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/merged_icesat2_final.parquet",
        compute=True
    )

[############################            ] | 72% Completed | 11.36 sms

In [ ]:
needed_hand = [
    'time', 'hand_alos_dem', 'hand_aster_dem', 'hand_copernicus_dеm',
    'hand_fab_dem', 'hand_nasa_dem', 'hand_srtm_dem', 'hand_tan_dem'
]
needed_geomorph = [
    'time', 'alos_dem_geomorphon', 'alos_dem_landform', 'aster_dem_geomorphon', 'aster_dem_landform',
    'copernicus_dеm_geomorphon', 'copernicus_dеm_landform', 'fab_dem_geomorphon', 'fab_dem_landform',
    'nasa_dem_geomorphon', 'nasa_dem_landform', 'srtm_dem_geomorphon', 'srtm_dem_landform',
    'tan_dem_geomorphon', 'tan_dem_landform'
]

regions = [6.0, 2.0]

for reg in regions:
    print(f"Processing region {reg}...")
    # 1. Читаємо terrain із region-фільтром
    terrain_part = pd.read_parquet(
        '/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/icesat2_with_terrain_attributes.parquet',
        filters=[('region', '==', reg)]
    )
    # 2. Читаємо hand і geomorphons — без filters!
    hand_part = pd.read_parquet(
        '/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/HAND_only_needed.parquet'
    )
    geomorph_part = pd.read_parquet(
        '/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/geomorphons_only_needed.parquet'
    )

    # Join
    result = terrain_part.set_index('time')
    result = result.join(hand_part.set_index('time'), how='left')
    result = result.join(geomorph_part.set_index('time'), how='left')
    result = result.reset_index()
    result.to_parquet(f'/mnt/c/Users/5302/OneDrive/PhD/paper_DEM_artickle/data/data_icesat2/merged_region_{int(reg)}.parquet')

print("Done!")



Processing region 6.0...
